# 03. Дерево решений

## Только классификация

Дерево решений — алгоритм, который последовательно задаёт вопросы о признаках и приводит объект к листу, где принимается решение о его классе.

В этом ноутбуке разберём:

- как дерево строится;
- как выбирается признак для разбиения;
- энтропию и Gini impurity;
- простой пример обучения;
- визуализацию дерева и границы классификации;
- псевдоалгоритм;
- гиперпараметры;
- преимущества и недостатки.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")

## 1. Интуиция

Представим кредитный скоринг только как иллюстрацию механики классификации.

Дерево может задавать вопросы:

- `доход > 100 000?`
- `возраст > 30?`
- `есть недвижимость?`

Каждый вопрос разделяет объекты на группы. В конце пути находится **лист**, которому назначается класс.

В отличие от KNN дерево во время обучения строит явную структуру правил.

## 2. Как дерево выбирает разбиение?

На каждом узле есть множество объектов. Желательно выбрать такой признак и такой порог, чтобы после разделения дочерние группы стали как можно более «чистыми».

Для оценки неоднородности часто используют **Gini impurity** или **энтропию**.

### Gini impurity

$$
Gini=1-\sum_k p_k^2.
$$

Если в узле находятся объекты только одного класса, `Gini = 0`.

Если классы перемешаны, значение становится больше.

### Энтропия

$$
H=-\sum_kp_k\log_2p_k.
$$

Чем сильнее перемешаны классы, тем выше энтропия.

In [ ]:
p = np.linspace(0.001, 0.999, 500)
gini = 2*p*(1-p)
entropy = -p*np.log2(p) - (1-p)*np.log2(1-p)

plt.figure(figsize=(8,5))
plt.plot(p, gini, label="Gini")
plt.plot(p, entropy, label="Entropy")
plt.xlabel("Доля объектов класса 1")
plt.ylabel("Неоднородность")
plt.title("Критерии качества разбиения для бинарной классификации")
plt.legend()
plt.show()

## 3. Простой пример обучения

Возьмём два признака и два класса.

Пусть дерево рассматривает множество возможных порогов:

$$
X_1 < t.
$$

Для каждого порога оно оценивает, насколько чистыми стали две получившиеся группы. Затем выбирается разбиение с наибольшим уменьшением неоднородности.

После первого разбиения процесс повторяется рекурсивно для дочерних узлов.

In [ ]:
X, y = make_moons(n_samples=250, noise=0.20, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)
tree.fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, tree.predict(X_test)))

## 4. Визуализация самого дерева

Каждый внутренний узел содержит условие, например `X1 <= 0.5`.  
Каждая ветвь соответствует результату этого условия.  
В листе хранится итоговое решение о классе.

In [ ]:
plt.figure(figsize=(16,8))
plot_tree(
    tree,
    feature_names=["X1", "X2"],
    class_names=["Класс 0", "Класс 1"],
    filled=True,
    rounded=True
)
plt.title("Обученное дерево классификации")
plt.show()

## 5. Визуализация границы классификации

Дерево решений с бинарными разбиениями по отдельным признакам создаёт области, ограниченные вертикальными и горизонтальными линиями.

In [ ]:
x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
xx, yy = np.meshgrid(
    np.linspace(x_min,x_max,500),
    np.linspace(y_min,y_max,500)
)

grid = np.c_[xx.ravel(), yy.ravel()]
z = tree.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8,6))
plt.contourf(xx, yy, z, alpha=0.20)
plt.scatter(X[y==0,0], X[y==0,1], label="Класс 0", edgecolor="black")
plt.scatter(X[y==1,0], X[y==1,1], label="Класс 1", edgecolor="black")
plt.xlabel("X1")
plt.ylabel("X2")
plt.title("Граница классификации дерева решений")
plt.legend()
plt.show()

## 6. Что будет, если не ограничивать глубину?

Если разрешить дереву продолжать деления почти до полного запоминания обучающей выборки, оно может получить очень сложную границу.

Это приводит к **переобучению**:

- ошибка на обучающей выборке становится очень маленькой;
- качество на новых данных может ухудшаться.

Поэтому глубина дерева и минимальный размер узлов — важные средства контроля сложности.

In [ ]:
depths = [1, 2, 3, 5, 10, None]
train_scores = []
test_scores = []

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))

pd.DataFrame({
    "max_depth": depths,
    "train_accuracy": train_scores,
    "test_accuracy": test_scores
})

## 7. Гиперпараметры дерева решений

### `max_depth`
Максимальная глубина дерева. Один из главных способов борьбы с переобучением.

### `min_samples_split`
Минимальное количество объектов, необходимое для попытки разделить внутренний узел.

### `min_samples_leaf`
Минимальное количество объектов в листе. Большое значение делает дерево более гладким.

### `max_leaf_nodes`
Ограничивает количество листьев.

### `criterion`
Критерий качества разбиения: например, `gini` или `entropy`.

### `max_features`
Ограничивает число признаков, рассматриваемых при поиске разбиения.

### `class_weight`
Позволяет учитывать различную стоимость ошибок по классам или дисбаланс классов.

## 8. Псевдоалгоритм построения дерева

Упрощённая схема:

1. В текущем узле находятся обучающие объекты.
2. Перебираем признаки.
3. Для каждого признака перебираем возможные пороги.
4. Для каждого разбиения считаем неоднородность дочерних узлов.
5. Выбираем лучшее разбиение.
6. Повторяем процедуру для дочерних узлов.
7. Останавливаемся, когда выполнено условие остановки: достигнута максимальная глубина, слишком мало объектов и т. п.
8. В листе прогнозируем наиболее частый класс.

In [ ]:
def pseudo_tree_split(X, y):
    best_split = None
    best_score = float("inf")

    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])

        for threshold in thresholds:
            left = X[:, feature] <= threshold
            right = ~left

            if left.sum() == 0 or right.sum() == 0:
                continue

            # Упрощённая оценка: взвешенная Gini impurity
            def gini(labels):
                if len(labels) == 0:
                    return 0
                _, counts = np.unique(labels, return_counts=True)
                p = counts / len(labels)
                return 1 - np.sum(p**2)

            score = (
                left.sum() / len(y) * gini(y[left])
                + right.sum() / len(y) * gini(y[right])
            )

            if score < best_score:
                best_score = score
                best_split = (feature, threshold)

    return best_split, best_score

## 9. Достоинства

- Легко объяснять человеку через правила «если — то».
- Не требует стандартизации признаков.
- Может моделировать нелинейные границы.
- Может автоматически выбирать важные признаки для разбиений.
- Подходит для числовых и, после соответствующей подготовки, категориальных признаков.
- Хорошо визуализируется на небольших деревьях.

## 10. Недостатки

- Глубокие деревья легко переобучаются.
- Небольшое изменение данных может привести к совершенно другой структуре дерева.
- Жадный выбор локально лучшего разбиения не гарантирует глобально оптимального дерева.
- Одинокое дерево часто уступает ансамблям вроде Random Forest или Gradient Boosting.
- При бинарных осевых разбиениях границы могут выглядеть ступенчато.

**Главная идея:** дерево решений превращает классификацию в последовательность простых вопросов, но сложность этих вопросов необходимо контролировать.